In [22]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300

In [23]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst_df = sst.to_dataframe().reset_index()

In [24]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [25]:
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [26]:
sst_df_jan = sst_df.query('month == 1')

In [27]:
chirps_eastern_east_africa = chirps.sel(latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

In [28]:
chirps_eastern_east_africa_jan = chirps_eastern_east_africa.query('month == 1').groupby(['month', 'year']).mean('precip').reset_index()

In [35]:
sst_df_jan

,lat,lon,time,nbnds,time_bnds,sst,month,year
3336,88.0,0.0,1993-01-01,0,9.969210e+36,-1.8,1,1993
3337,88.0,0.0,1993-01-01,1,9.969210e+36,-1.8,1,1993
3360,88.0,0.0,1994-01-01,0,9.969210e+36,-1.8,1,1994
3361,88.0,0.0,1994-01-01,1,9.969210e+36,-1.8,1,1994
3384,88.0,0.0,1995-01-01,0,9.969210e+36,-1.8,1,1995
...,...,...,...,...,...,...,...,...
65778047,-88.0,358.0,2022-01-01,1,9.969210e+36,NaN,1,2022
65778070,-88.0,358.0,2023-01-01,0,9.969210e+36,NaN,1,2023
65778071,-88.0,358.0,2023-01-01,1,9.969210e+36,NaN,1,2023
65778094,-88.0,358.0,2024-01-01,0,9.969210e+36,NaN,1,2024


In [29]:
start_year = 1993
end_year = 2024

# Create a boolean mask
mask = (chirps_eastern_east_africa_jan['year'] >= start_year) & (chirps_eastern_east_africa_jan['year'] <= end_year)

# Apply the mask to filter the DataFrame
chirps_eastern_east_africa_jan = chirps_eastern_east_africa_jan[mask]

mask = (sst_df_jan['year'] >= start_year) & (sst_df_jan['year'] <= end_year)

sst_df_jan = sst_df_jan[mask]

In [30]:
chirps_eastern_east_africa_jan.quantile([0.33, 0.66])

,month,year,latitude,longitude,precip
0.33,1.0,2003.23,2.249998,43.999996,4.167859
0.66,1.0,2013.46,2.249998,43.999996,6.129906


In [31]:
jan_bn = chirps_eastern_east_africa_jan.query('precip <= 4.16786')

In [32]:
jan_n = chirps_eastern_east_africa_jan.query('4.16786 <= precip <= 6.12991')

In [33]:
jan_an = chirps_eastern_east_africa_jan.query('6.12991 <= precip')

In [34]:
jan_an['year'].to_list()

[1993, 1996, 1998, 2001, 2002, 2004, 2008, 2009, 2016, 2020, 2022]